In [22]:
import glob
import json
import os
import sys
import time
from collections import deque
from pathlib import Path

from dataclasses import dataclass, field
from typing import Any, Iterable, Optional
import cudf
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import seaborn as sns
import shap
import wandb
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

sys.path.append(os.path.abspath(".."))

from src.utils.print_duration import print_duration
import src.models.xgb.xgb_cv_trainer as cv

In [ ]:
import numpy as np, pyarrow.dataset as ds, torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info


@dataclass(eq=False)
class ParquetStream(IterableDataset):
    # === 引数（元 __init__ のシグネチャ） ===
    paths: list[str] | str | os.PathLike
    features: Optional[list[str]] = None
    target: str = "target"
    cat_cols: Optional[Iterable[str]] = None
    fold_col: Optional[str] = None
    include_folds: Optional[Iterable[int]] = None
    exclude_folds: Optional[Iterable[int]] = None
    weight_col: Optional[str] = None
    batch_rows: int = 200_000
    buffer_size: int = 100_000
    rows_per_epoch: int = None
    extra_exclude_cols: Optional[Iterable[str]] = None
    predict_mode: bool = False
    seed: int = 42

    _epoch: int = field(init=False, default=0, repr=False)

    def __post_init__(self):
        super().__init__()
        self.paths = [
            str(p)
            for p in (
                self.paths
                if isinstance(self.paths, (list, tuple))
                else [self.paths]
            )
        ]
        self.buffer_size = int(self.buffer_size)
        self.cat_cols = list(self.cat_cols or [])
        self.include_folds = (
            None
            if self.include_folds is None
            else set(self.include_folds)
        )
        self.exclude_folds = (
            None
            if self.exclude_folds is None
            else set(self.exclude_folds)
        )
        self.predict_mode = bool(self.predict_mode)
        self.keep_row_ids = bool(self.keep_row_ids)

        # --- extra exclude colsを除外 ---
        schema = ds.dataset(self.paths, format="parquet").schema
        all_cols = [f.name for f in schema]

        if self.extra_exclude_cols:
            excl = (
                {self.extra_exclude_cols}
                if isinstance(self.extra_exclude_cols, str)
                else set(self.extra_exclude_cols)
            )
            self.features = [c for c in self.features if c not in excl]

        # 入力列（重複除去）
        cols = list(self.features)
        if (not self.predict_mode) and (self.target in all_cols):
            cols.append(self.target)
        if (not self.predict_mode) and (self.weight_col in all_cols):
            cols.append(self.weight_col)
        if (not self.predict_mode) and (self.fold_col in all_cols):
            cols.append(self.fold_col)
        cols.append("row_id")
        self._columns = list(dict.fromkeys(cols))

        # 内部状態
        self._reader = None
        self._current_file_index = 0

    def set_epoch(self, epoch: int):
        self._epoch = int(epoch)

    def _sharded_paths(self):
        info = get_worker_info()
        if info is None:
            return self.paths
        return self.paths[info.id::info.num_workers]

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self._epoch + (get_worker_info().id if get_worker_info() else 0))
        bufX, bufy, bufw = [], [], []
        emitted = 0
        fexpr = None
        if (not self.predict_mode) and self.fold_col:
            col = ds.field(self.fold_col)
            if self.include_folds is not None:
                fexpr = col.isin(sorted(self.include_folds))
            if self.exclude_folds is not None:
                ex = ~col.isin(sorted(self.exclude_folds))
                fexpr = ex if fexpr is None else (fexpr & ex)

        for path in self._sharded_paths():
            reader = ds.dataset(path, format="parquet").scanner(
                columns=self._columns,
                batch_size=self.batch_rows
            ).to_reader()
            for batch in reader:
                tbl = batch.to_pandas()  # 速さ重視なら .to_numpy() 直取りでもOK
                X = tbl[self.features].to_numpy(dtype=np.float32, copy=False)
                y = tbl[self.target_col].to_numpy(dtype=np.float32, copy=False)
                w = tbl[self.weight_col].to_numpy(dtype=np.float32, copy=False) if self.weight_col else None

                # バッファに積む
                bufX.append(X); bufy.append(y); 
                if self.weight_col: bufw.append(w)
                # バッファが大きくなり過ぎたら1つにまとめてシャッフル
                if sum(len(a) for a in bufy) >= self.buffer_size:
                    Xb = np.concatenate(bufX); yb = np.concatenate(bufy)
                    wb = np.concatenate(bufw) if self.weight_col else None
                    idx = rng.permutation(len(yb))
                    Xb, yb = Xb[idx], yb[idx]
                    if wb is not None: wb = wb[idx]
                    # まとめてyield
                    for i in range(len(yb)):
                        if self.rows_per_epoch and emitted >= self.rows_per_epoch:
                            return
                        xb = torch.from_numpy(Xb[i])
                        ybt = torch.tensor(yb[i])
                        if wb is None:
                            yield xb, ybt
                        else:
                            yield xb, ybt, torch.tensor(wb[i])
                        emitted += 1
                    bufX.clear(); bufy.clear(); bufw.clear()
        # 余りを吐く
        if bufy:
            Xb = np.concatenate(bufX); yb = np.concatenate(bufy)
            wb = np.concatenate(bufw) if self.weight_col else None
            idx = rng.permutation(len(yb))
            Xb, yb = Xb[idx], yb[idx]
            if wb is not None: wb = wb[idx]
            for i in range(len(yb)):
                if self.rows_per_epoch and emitted >= self.rows_per_epoch:
                    return
                xb = torch.from_numpy(Xb[i])
                ybt = torch.tensor(yb[i])
                if wb is None:
                    yield xb, ybt
                else:
                    yield xb, ybt, torch.tensor(wb[i])


In [37]:
import copy
import optuna
from optuna.trial import create_trial
from optuna.trial import TrialState
from dotenv import load_dotenv
import copy
import optuna
from optuna.trial import create_trial, TrialState



def bulk_clone_studies(rename_map: dict[str, str], storage: str) -> None:
    """
    {旧study名: 新study名} を一括でクローン（実質リネーム）する。
    - 単目的/多目的の違いを見て create_trial に value / values を正しく渡す
    - RUNNING trial はスキップ
    - study / trial の user_attrs をコピー
    """
    for old_name, new_name in rename_map.items():
        print(f"[INFO] Cloning study '{old_name}' -> '{new_name}'")

        # 元study
        src = optuna.load_study(study_name=old_name, storage=storage)
        is_multi = len(src.directions) > 1

        # sampler / pruner / directions を引き継いで新study作成
        sampler = copy.deepcopy(src.sampler)
        pruner = copy.deepcopy(src.pruner)
        if is_multi:
            dst = optuna.create_study(
                study_name=new_name,
                storage=storage,
                directions=src.directions,
                sampler=sampler,
                pruner=pruner,
                load_if_exists=False,
            )
        else:
            dst = optuna.create_study(
                study_name=new_name,
                storage=storage,
                direction=src.direction,
                sampler=sampler,
                pruner=pruner,
                load_if_exists=False,
            )

        # Studyのuser_attrsもコピー
        for k, v in src.user_attrs.items():
            dst.set_user_attr(k, copy.deepcopy(v))

        # trialをコピー（RUNNINGはスキップ）
        count = 0
        for t in src.get_trials(deepcopy=True):
            if t.state == TrialState.RUNNING:
                continue

            kwargs = dict(
                state=t.state,
                params=t.params,
                distributions=t.distributions,
                intermediate_values=t.intermediate_values,
                user_attrs=t.user_attrs,
            )
            # 単目的/多目的で value / values を切り替え
            if is_multi:
                kwargs["values"] = t.values  # list[float] | None
            else:
                kwargs["value"] = t.value    # float | None

            ft = create_trial(**kwargs)
            dst.add_trial(ft)
            count += 1

        print(f"[DONE] {old_name} -> {new_name} (trials_copied={count})")


In [38]:
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

url = os.environ.get("OPTUNA_STORAGE_URL")

In [42]:
rename_dict = {
    "cb_v1": "cb-001",
    "cb_v2": "cb-012",
    "lgbm_v1": "lgbm-001",
    "lgbm_v2": "lgbm-012",
    "lgbm_v3": "lgbm-023",
    "logreg_v1": "logreg-009",
    "logreg_v2": "logreg-017",
    "logreg_v3": "logreg-020",
    "mlp_v1": "mlp-002",
    "mlp_v2": "mlp-004",
    "mlp_v3": "mlp-005",
    "mlp_v4": "mlp-006",
    "mlp_v5": "mlp-007",
    "mlp_v6": "mlp-008",
    "mlp_v7": "mlp-010",
    "mlp_v8": "mlp-020",
    "mlp_v9": "mlp-024",
    "rfc_v1": "rfc-001",
    "rfc_v2": "rfc-012",
    "rfc_v3": "rfc-019",
    "rfc_v4": "rfc-021",
    "xgb_v1": "xgb-001",
    "xgb_v2": "xgb-003",
    "xgb_v3": "xgb-011",
    "xgb_v4": "xgb-012",
    "xgb_v5": "xgb-015",
    "xgb_v6": "xgb-013",
    "xgb_v7": "xgb-016",
    "xgb_v8": "xgb-019",
    "xgb_v9": "xgb-021",
}
# bulk_clone_studies(rename_dict, url)

In [43]:
for study in rename_dict.keys():
    optuna.delete_study(
        study_name=study, storage=url)